# Market Risk Analytics in Python

## Project objective

The objective of this project is to build a Python-based market risk analytics workflow for a multi-asset portfolio.

The project will measure historical risk, downside risk, diversification, stress scenario impact, and simulated future outcomes using daily market data.

This project is for educational and analytical purposes only and is not investment advice.

## Step 1: Define the portfolio

For the first version of this project, I will analyze a simplified multi-asset ETF portfolio.

The portfolio includes exposure to:

- U.S. large-cap equities
- U.S. growth equities
- U.S. small-cap equities
- Developed international equities
- Emerging-market equities
- U.S. investment-grade bonds
- Gold

The goal is not to create the perfect investment portfolio. The goal is to create a realistic portfolio that allows me to study risk, diversification, volatility, drawdowns, VaR, CVaR, stress testing, and Monte Carlo simulation.

In [ ]:
import numpy as np
import pandas as pd

tickers = ["SPY", "QQQ", "IWM", "EFA", "EEM", "AGG", "GLD"]

weights = np.array([0.30, 0.20, 0.10, 0.10, 0.10, 0.15, 0.05])

portfolio = pd.DataFrame({
    "Ticker": tickers,
    "Weight": weights,
    "Weight (%)": weights * 100,
    "Role": [
        "U.S. large-cap equity exposure",
        "U.S. growth and technology exposure",
        "U.S. small-cap equity exposure",
        "Developed international equity exposure",
        "Emerging-market equity exposure",
        "U.S. investment-grade bond exposure",
        "Gold / alternative defensive exposure"
    ]
})

portfolio

In [ ]:
if not np.isclose(weights.sum(), 1.0):
    raise ValueError("Portfolio weights must sum to 1.")

print(f"Total portfolio weight: {weights.sum():.2%}")

## Step 2: Choose the analysis period

For this project, I will use daily historical data from 2019-01-01 to the latest available market data at the time the notebook is run.

This period was selected because it includes multiple market environments, including the COVID-19 crash, the post-COVID recovery, the 2022 inflation and interest-rate shock, the 2023–2024 recovery period, and the most recent market conditions.

Because the end date updates when the notebook is rerun, the results may change over time. This makes the project more realistic as a living market risk analytics workflow.

In [ ]:
start_date = "2019-01-01"
end_date = None  # Use latest available market data

print(f"Analysis period starts on: {start_date}")
print("Analysis period ends at the latest available market data.")

## Step 3: Download historical price data

In this step, I download daily historical adjusted closing prices for all selected ETFs.

Adjusted prices are used because they account for corporate actions such as dividends and splits, making them more appropriate for return and risk calculations.

In [ ]:
import yfinance as yf

price_data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

prices = price_data["Close"]

prices.head()

In [ ]:
print("Price data shape:", prices.shape)
print("Start date:", prices.index.min())
print("End date:", prices.index.max())

prices.tail()

In [ ]:
missing_values = pd.DataFrame({
    "Missing Values": prices.isna().sum(),
    "Missing (%)": prices.isna().mean() * 100
})

missing_values

In [ ]:
prices = prices.dropna()

print("Cleaned price data shape:", prices.shape)
print("Cleaned start date:", prices.index.min())
print("Cleaned end date:", prices.index.max())

## Step 4: Calculate daily returns

In this step, I convert adjusted closing prices into daily percentage returns.

Risk modeling is based on returns rather than raw prices because returns measure the relative change in value over time.

For the first version of this project, I will use simple daily returns instead of log returns because they are easier to interpret and communicate.

In [ ]:
returns = prices.pct_change().dropna()

returns.head()

In [ ]:
print("Returns data shape:", returns.shape)
print("Start date:", returns.index.min())
print("End date:", returns.index.max())

returns.describe()

In [ ]:
portfolio_returns = returns @ weights

portfolio_returns.head()

In [ ]:
portfolio_returns_summary = pd.DataFrame({
    "Metric": [
        "Mean Daily Return",
        "Daily Volatility",
        "Minimum Daily Return",
        "Maximum Daily Return"
    ],
    "Value": [
        portfolio_returns.mean(),
        portfolio_returns.std(),
        portfolio_returns.min(),
        portfolio_returns.max()
    ]
})

portfolio_returns_summary

In [ ]:
cumulative_returns = (1 + portfolio_returns).cumprod()

cumulative_returns.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(cumulative_returns)
plt.title("Portfolio Growth Over Time")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.grid(True)
plt.show()

## Step 5: Measure historical portfolio risk

In this step, I calculate the main historical risk metrics for the portfolio:

- Annualized return
- Annualized volatility
- Sharpe ratio
- Maximum drawdown
- Historical Value at Risk (VaR)
- Historical Conditional Value at Risk (CVaR)

These metrics help quantify return, risk, downside losses, and extreme loss behavior.

In [ ]:
trading_days = 252
annual_risk_free_rate = 0.00  # Baseline assumption for first version

annualized_return = (1 + portfolio_returns.mean()) ** trading_days - 1
annualized_volatility = portfolio_returns.std() * np.sqrt(trading_days)

sharpe_ratio = (annualized_return - annual_risk_free_rate) / annualized_volatility

annualized_return, annualized_volatility, sharpe_ratio

In [ ]:
running_max = cumulative_returns.cummax()
drawdown = cumulative_returns / running_max - 1

max_drawdown = drawdown.min()

max_drawdown

In [ ]:
confidence_level = 0.95

var_95 = portfolio_returns.quantile(1 - confidence_level)
cvar_95 = portfolio_returns[portfolio_returns <= var_95].mean()

var_95, cvar_95

In [ ]:
risk_summary = pd.DataFrame({
    "Metric": [
        "Annualized Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown",
        "Historical VaR 95%",
        "Historical CVaR 95%"
    ],
    "Value": [
        annualized_return,
        annualized_volatility,
        sharpe_ratio,
        max_drawdown,
        var_95,
        cvar_95
    ]
})

risk_summary

In [ ]:
risk_summary_display = risk_summary.copy()

risk_summary_display["Value"] = risk_summary_display.apply(
    lambda row: f"{row['Value']:.2f}" if row["Metric"] == "Sharpe Ratio" else f"{row['Value']:.2%}",
    axis=1
)

risk_summary_display

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(drawdown)
plt.title("Portfolio Drawdown Over Time")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.grid(True)
plt.show()

### Initial interpretation

The annualized return shows the average yearly growth rate of the portfolio over the selected historical period.

The annualized volatility measures how much the portfolio returns fluctuate over time.

The Sharpe ratio compares return to risk. In this first version, I assume a 0% risk-free rate as a baseline assumption.

Maximum drawdown shows the worst peak-to-trough decline experienced by the portfolio.

Historical VaR at 95% estimates the daily loss threshold that was only exceeded in the worst 5% of historical days.

Historical CVaR at 95% estimates the average loss on days when losses were worse than the VaR threshold.